In [29]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [30]:
results_df = pd.read_csv("results.csv")
rankings_df = pd.read_csv("fifa_ranking-2024-06-20.csv")

EDA & preprocessing

In [31]:
results_df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [32]:
rankings_df.head()

,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,140.0,Brunei Darussalam,BRU,2.0,0.0,140,AFC,1992-12-31
1,33.0,Portugal,POR,38.0,0.0,33,UEFA,1992-12-31
2,32.0,Zambia,ZAM,38.0,0.0,32,CAF,1992-12-31
3,31.0,Greece,GRE,38.0,0.0,31,UEFA,1992-12-31
4,30.0,Algeria,ALG,39.0,0.0,30,CAF,1992-12-31


In [33]:
results_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49477 entries, 0 to 49476
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        49477 non-null  str    
 1   home_team   49477 non-null  str    
 2   away_team   49477 non-null  str    
 3   home_score  49433 non-null  float64
 4   away_score  49433 non-null  float64
 5   tournament  49477 non-null  str    
 6   city        49477 non-null  str    
 7   country     49477 non-null  str    
 8   neutral     49477 non-null  bool   
dtypes: bool(1), float64(2), str(6)
memory usage: 3.1 MB


In [34]:
rankings_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 67472 entries, 0 to 67471
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   rank             67463 non-null  float64
 1   country_full     67472 non-null  str    
 2   country_abrv     67472 non-null  str    
 3   total_points     67472 non-null  float64
 4   previous_points  67472 non-null  float64
 5   rank_change      67472 non-null  int64  
 6   confederation    67472 non-null  str    
 7   rank_date        67472 non-null  str    
dtypes: float64(3), int64(1), str(4)
memory usage: 4.1 MB


In [35]:
results_df["date"] = pd.to_datetime(results_df["date"])
rankings_df["rank_date"] = pd.to_datetime(rankings_df["rank_date"])

In [36]:
results_df["tournament"].value_counts()

tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
                                        ...  
Copa Confraternidad                         1
Benedikt Fontana Cup                        1
ConIFA Challenger Cup                       1
CONIFA World Cup qualification              1
South Asian Super Cup                       1
Name: count, Length: 200, dtype: int64

In [37]:
competitive = ["FIFA World Cup qualification", "UEFA Euro qualification", "African Cup of Nations qualification",
               "FIFA World Cup", "Copa América", "African Cup of Nations", "AFC Asian Cup qualification",
               "UEFA Nations League" , "CECAFA Cup"]

results_df = results_df[results_df['tournament'].isin(competitive)]

In [38]:
results_df.shape

(18779, 9)

In [39]:
def get_results(row):
    if row["home_score"] > row["away_score"]:
        return 2 #home win
    elif row["home_score"] == row["away_score"]:
        return 1 #draw
    else:
        return 0 #away win

results_df["result"] = results_df.apply(get_results, axis=1) #create a new results row applying the match win result

In [54]:
#merge home team rankings to results dataframe in a new dataframe called df_home
df_home = pd.merge_asof(
    results_df.sort_values('date'),
    rankings_df[['rank_date','country_full','rank','total_points']].rename(columns={
        'rank_date'    : 'date',
        'country_full' : 'home_team',
        'rank'         : 'home_rank',
        'total_points' : 'home_points'
    }).sort_values('date'),
    on='date',
    by='home_team'
)


In [55]:
#merge away team rankings
df_final = pd.merge_asof(
    df_home.sort_values('date'),
    rankings_df[['rank_date','country_full','rank','total_points']].rename(columns={
        'rank_date'    : 'date',
        'country_full' : 'away_team',
        'rank'         : 'away_rank',
        'total_points' : 'home_points'
    }).sort_values('date'),
    on='date',
    by='away_team'
)

In [41]:
df_final.shape

(18779, 14)

In [42]:
df_final.head(10) #first 10 rows

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result,home_rank,home_points_x,away_rank,home_points_y
0,1916-07-02,Chile,Uruguay,0.0,4.0,Copa América,Buenos Aires,Argentina,True,0,NaN,NaN,NaN,NaN
1,1916-07-06,Argentina,Chile,6.0,1.0,Copa América,Buenos Aires,Argentina,False,2,NaN,NaN,NaN,NaN
2,1916-07-08,Brazil,Chile,1.0,1.0,Copa América,Buenos Aires,Argentina,True,1,NaN,NaN,NaN,NaN
3,1916-07-10,Argentina,Brazil,1.0,1.0,Copa América,Buenos Aires,Argentina,False,1,NaN,NaN,NaN,NaN
4,1916-07-12,Brazil,Uruguay,1.0,2.0,Copa América,Buenos Aires,Argentina,True,0,NaN,NaN,NaN,NaN
5,1916-07-17,Argentina,Uruguay,0.0,0.0,Copa América,Avellaneda,Argentina,False,1,NaN,NaN,NaN,NaN
6,1917-09-30,Uruguay,Chile,4.0,0.0,Copa América,Montevideo,Uruguay,False,2,NaN,NaN,NaN,NaN
7,1917-10-03,Argentina,Brazil,4.0,2.0,Copa América,Montevideo,Uruguay,True,2,NaN,NaN,NaN,NaN
8,1917-10-06,Argentina,Chile,1.0,0.0,Copa América,Montevideo,Uruguay,True,2,NaN,NaN,NaN,NaN
9,1917-10-07,Uruguay,Brazil,4.0,0.0,Copa América,Montevideo,Uruguay,False,2,NaN,NaN,NaN,NaN


In [43]:
df_final[["home_rank", "away_rank"]].isna().sum()

home_rank    6106
away_rank    6170
dtype: int64

In [44]:
post_1992 = df_final[df_final["date"] > '1992-01-01']
print(post_1992[['date','home_team','away_team','home_rank','away_rank']].head())

           date home_team    away_team  home_rank  away_rank
4797 1992-01-12  Cameroon      Morocco        NaN        NaN
4798 1992-01-12   Senegal      Nigeria        NaN        NaN
4799 1992-01-13   Algeria  Ivory Coast        NaN        NaN
4800 1992-01-13     Egypt       Zambia        NaN        NaN
4801 1992-01-14   Morocco     DR Congo        NaN        NaN


In [45]:
# Check what team names look like in each CSV
print("results.csv sample teams:")
print(results_df['home_team'].unique()[:20])



results.csv sample teams:
<StringArray>
[              'Chile',           'Argentina',              'Brazil',
             'Uruguay',            'Paraguay',             'Bolivia',
                'Peru',             'Belgium',              'France',
              'Sweden',           'Lithuania',          'Yugoslavia',
              'Poland',         'Switzerland',               'Haiti',
 'Republic of Ireland',              'Mexico',          'Luxembourg',
               'Spain',               'Egypt']
Length: 20, dtype: str


In [46]:
print("\nfifa_ranking.csv sample teams:")
print(rankings_df['country_full'].unique()[:20])


fifa_ranking.csv sample teams:
<StringArray>
['Brunei Darussalam',          'Portugal',            'Zambia',
            'Greece',           'Algeria',        'Yugoslavia',
             'Wales',     'Côte d'Ivoire',           'Austria',
          'Bulgaria',               'USA',          'Scotland',
          'Cameroon',             'Egypt',            'Poland',
            'France',    'Czechoslovakia',            'Mexico',
          'Colombia',           'Hungary']
Length: 20, dtype: str


In [51]:
# there's a mismatch between names of our two csv files which might have caused the NaN in home nd away rank
name_fix = {
    'United States'       : 'USA',
    'Ivory Coast'         : 'Côte d\'Ivoire',
    'Republic of Ireland' : 'Ireland',
    'South Korea'         : 'Korea Republic',
    'North Korea'         : 'Korea DPR',
    'Czech Republic'      : 'Czechia',
    'Macedonia'           : 'North Macedonia',
    'Swaziland'           : 'Eswatini',
    'Cape Verde'          : 'Cape Verde Islands',
}

results_df['home_team'] = results_df['home_team'].replace(name_fix)
results_df['away_team'] = results_df['away_team'].replace(name_fix)

In [56]:
post_1992 = df_final[df_final['date'] > '1992-01-01']
print(post_1992[['date','home_team','away_team','home_rank','away_rank']].head(10))

           date      home_team      away_team  home_rank  away_rank
4797 1992-01-12       Cameroon        Morocco        NaN        NaN
4798 1992-01-12        Senegal        Nigeria        NaN        NaN
4799 1992-01-13        Algeria  Côte d'Ivoire        NaN        NaN
4800 1992-01-13          Egypt         Zambia        NaN        NaN
4801 1992-01-14        Morocco       DR Congo        NaN        NaN
4802 1992-01-14        Nigeria          Kenya        NaN        NaN
4803 1992-01-15  Côte d'Ivoire          Congo        NaN        NaN
4804 1992-01-15         Zambia          Ghana        NaN        NaN
4805 1992-01-16       Cameroon       DR Congo        NaN        NaN
4806 1992-01-16        Senegal          Kenya        NaN        NaN
